In [1]:
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['Helvetica'] + matplotlib.rcParams['font.sans-serif']
matplotlib.rcParams['font.size'] = 6
matplotlib.rcParams['text.usetex'] = False
matplotlib.rcParams["ps.usedistiller"] = 'xpdf'
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.weight'] = 'normal'
matplotlib.rcParams["mathtext.fontset"] = 'cm'

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import random
import math

import pandas as pd


import copy

import cvxpy
cp = cvxpy

import figurefirst as fifi

from braid_analysis import braid_analysis_plots

In [3]:
import sys
from pathlib import Path


In [4]:
from align_course_direction_analysis import unifying_algo_analysis as uaa
from align_course_direction_analysis import unifying_algo_plots as uap

# Helper Functions

In [5]:
#from splitflow.unifying_algo_analysis_helper import *
from splitflow.set_zorder_functions import *

Using device: cpu


In [6]:
def mean_angle(angle):
    
    mean = np.arctan2( np.nanmean(np.sin(angle)), np.nanmean(np.cos(angle)) )
    return mean

def angle_distance(angle1, angle2):
    """
    Calculate the minimum distance between two angles.

    Parameters:
    -----------
    angle1, angle2 : float or array-like
        Angles in radians

    Returns:
    --------
    float or array
        Minimum distance between angles in radians 
        Range: [-π, π]
    """
    diff = angle1 - angle2
    # Wrap to [-π, π]
    distance = np.arctan2(np.sin(diff), np.cos(diff))
    return distance

In [7]:
def clean_labels(ax, show_labels, spines=['left', 'bottom']):
    #set_selective_rasterization(ax, rasterize_markers=['.'], raster_zorder=-1)
    #set_selective_rasterization(ax, rasterize_collections=[mcollections.PolyCollection], raster_zorder=-2)
    ax.set_rasterization_zorder(0)

    ax.set_yticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
    ax.set_ylim(-np.pi, np.pi)
    ax.set_xlim(-2, 2)
    ax.set_xticks([-200, -100, 0, 100, 200])

    if show_labels:
        ax.set_xticklabels(['-2', '-1', '0', '1', '2'])
    else:
        ax.set_xticklabels([])
    
    if show_labels:
        ax.set_yticklabels(['$-\pi$', '','$0$','','$\pi$',])
    else:
        ax.set_yticklabels([])
        
    fifi.mpl_functions.adjust_spines(ax, ['left', 'bottom'])
    
    if show_labels:
        ax.set_ylabel('Course direction', labelpad=-2)
        ax.set_xlabel('Aligned time (s)', labelpad=1)
    else:
        ax.set_ylabel('')
        ax.set_xlabel('')
    
    ax.tick_params(axis='y', pad=2)
    ax.tick_params(axis='x', pad=2)
    
    fifi.mpl_functions.adjust_spines(ax, ['left', 'bottom'],
                                     tick_length=2.5,
                                     spine_locations={'left': 5, 'bottom': 5},
                                     linewidth=0.5)
    fifi.mpl_functions.set_fontsize(ax, 6)

In [8]:
FIGURE_NAME = 'supplemental_unifying_analysis_variable_wind_unifying.svg'

In [9]:
TRANSLATION = True

In [10]:
COURSE_MARKER_SIZE = 2
COURSE_ALPHA_MULTIPLIER = 3

In [11]:
class LabelToMetadata:
    def __init__(self):
        condition = 'flash'
        self.flash = {str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/varwind_N700_' + condition + '_12.0_translation' + str(TRANSLATION) + '.parquet': [1, 'gray', '12', 0], 
                      str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/varwind_N700_' + condition + '_20.0_translation' + str(TRANSLATION) + '.parquet': [2, 'gray', '20', 0.03],
                      str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/varwind_N700_' + condition + '_23.0_translation' + str(TRANSLATION) + '.parquet': [3, 'gray', '23', 0.1],
                      str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/varwind_N700_' + condition + '_28.0_translation' + str(TRANSLATION) + '.parquet': [4, '#a245ffff', '28', 0.2],
                      str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/varwind_N700_' + condition + '_40.0_translation' + str(TRANSLATION) + '.parquet': [5, 'gray', '40', 0.3],
                      str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/varwind_N700_' + condition + '_90.0_translation' + str(TRANSLATION) + '.parquet': [6, 'gray', '90', 0.6],
                       }
        condition = 'sham'
        self.sham = {str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/varwind_N700_' + condition + '_12.0_translation' + str(TRANSLATION) + '.parquet': [1, 'gray', '12', 0], 
                     str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/varwind_N700_' + condition + '_20.0_translation' + str(TRANSLATION) + '.parquet': [2, 'gray', '20', 0.03],
                     str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/varwind_N700_' + condition + '_23.0_translation' + str(TRANSLATION) + '.parquet': [3, 'gray', '23', 0.1],
                     str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/varwind_N700_' + condition + '_28.0_translation' + str(TRANSLATION) + '.parquet': [4, '#bd7bffff', '28', 0.2],
                     str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/varwind_N700_' + condition + '_40.0_translation' + str(TRANSLATION) + '.parquet': [5, 'gray', '40', 0.3],
                     str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/varwind_N700_' + condition + '_90.0_translation' + str(TRANSLATION) + '.parquet': [6, 'gray', '90', 0.6],
                               }

        # Temporary
        self.sham = {str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/var_wind_analyzed/' + 'var-wind-12-fan_0-cms_all-traj_align_SHAM' + '.parquet': [1, 'gray', '12', 0], 
                                 str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/var_wind_analyzed/' + 'var-wind-20-fan_0-03-cms_all-traj_align_SHAM' + '.parquet': [2, 'gray', '20', 0.03],
                                 str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/var_wind_analyzed/' + 'var-wind-23-fan_0-1-cms_all-traj_align_SHAM' + '.parquet': [3, '#bd7bffff', '23', 0.1],
                                 str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/var_wind_analyzed/' + 'var-wind-28-fan_0-2-cms_all-traj_align_SHAM' + '.parquet': [4, 'gray', '28', 0.2],
                                 str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/var_wind_analyzed/' + 'var-wind-40-fan_0-3-cms_all-traj_align_SHAM' + '.parquet': [5, 'gray', '40', 0.3],
                                 str(Path('../../../Data/Unifying_Algo_Results/Supplemental')) + '/var_wind_analyzed/' + 'var-wind-90-fan_0-6-cms_all-traj_align_SHAM' + '.parquet': [6, 'gray', '90', 0.6],
                               }

In [12]:
def get_filename_for_wind_type(metadata, windtype):
    filename = None
    for key, val in metadata.items():
        if windtype in val:
            filename = key
    return filename

In [13]:
def get_trajec_filename_from_unifying_filename(unifying_filename):
    return str(Path('../../../Data/Experimental_Fly_Data')) + '/flies_varwind_C1WT_combined_optotrigger_fancontrol_trimmed.parquet'


In [14]:
def get_filenames_for_metadata_windtype(metadata, windtype):
    unifying_filename = get_filename_for_wind_type(metadata, windtype)
    trajectory_filename = None
    df = None
    unifying_algo_data = None
    
    if unifying_filename != 'None':
        print(unifying_filename)
        unifying_algo_data = pd.read_parquet(unifying_filename)
    
        trajectory_filename = get_trajec_filename_from_unifying_filename(unifying_filename)
        if '.hdf' in trajectory_filename:
            df = pd.read_hdf(trajectory_filename)
        else:
            df = pd.read_parquet(trajectory_filename)
    
    else:
        unifying_algo_data = None

    print(unifying_filename)
    print(trajectory_filename)
    return unifying_algo_data, df

# Set up

Run notebook with label 'flash' and 'sham'

In [15]:
label = 'flash'
flash_or_sham = label
label_to_metadata = LabelToMetadata()
fifi_figure_label = 'unifying_' + label
metadata = label_to_metadata.__getattribute__(label)

In [16]:
metadata

{'../../../Data/Unifying_Algo_Results/Supplemental/varwind_N700_flash_12.0_translationTrue.parquet': [1,
  'gray',
  '12',
  0],
 '../../../Data/Unifying_Algo_Results/Supplemental/varwind_N700_flash_20.0_translationTrue.parquet': [2,
  'gray',
  '20',
  0.03],
 '../../../Data/Unifying_Algo_Results/Supplemental/varwind_N700_flash_23.0_translationTrue.parquet': [3,
  'gray',
  '23',
  0.1],
 '../../../Data/Unifying_Algo_Results/Supplemental/varwind_N700_flash_28.0_translationTrue.parquet': [4,
  '#a245ffff',
  '28',
  0.2],
 '../../../Data/Unifying_Algo_Results/Supplemental/varwind_N700_flash_40.0_translationTrue.parquet': [5,
  'gray',
  '40',
  0.3],
 '../../../Data/Unifying_Algo_Results/Supplemental/varwind_N700_flash_90.0_translationTrue.parquet': [6,
  'gray',
  '90',
  0.6]}

In [17]:
windtype = '12'
unifying_algo_data, df = get_filenames_for_metadata_windtype(metadata, windtype)

../../../Data/Unifying_Algo_Results/Supplemental/varwind_N700_flash_12.0_translationTrue.parquet


../../../Data/Unifying_Algo_Results/Supplemental/varwind_N700_flash_12.0_translationTrue.parquet
../../../Data/Experimental_Fly_Data/flies_varwind_C1WT_combined_optotrigger_fancontrol_trimmed.parquet


# variable wind data

In [18]:
layout = fifi.svg_to_axes.FigureLayout(FIGURE_NAME, autogenlayers=True, make_mplfigures=True, hide_layers=[], dpi=600)
plt.close('all')

In [19]:
show_labels = True

In [20]:
windtype = '12'
unifying_algo_data, df = get_filenames_for_metadata_windtype(metadata, windtype)
if unifying_algo_data is not None:
    ax = layout.axes[(fifi_figure_label, windtype)]
    
    # build aligned arrays
    results = uaa.get_aligned_course_array(unifying_algo_data, df, return_linear_and_affine_fits=True)
    ix_master, aligned_time_since_flash_arr, aligned_course_arr, aligned_roi_arr, aligned_linear_fits, aligned_affine_fits = results
    
    # build flash arrays from aligned time
    flash_length = 500 # ms
    aligned_flash_arr = np.zeros_like(aligned_time_since_flash_arr)
    ix = np.where( (aligned_time_since_flash_arr>0) * (aligned_time_since_flash_arr<flash_length/1000.) )
    aligned_flash_arr[ix] = 1

    
    # Mean affine fit
    mean_affine_fit = [mean_angle(aligned_affine_fits[:,i]) for i in range(aligned_affine_fits.shape[1])]
    mean_affine_fit = np.array(mean_affine_fit)
    mean_affine_fit[0:200] = np.nan
    mean_affine_fit[400:] = np.nan
    mean_affine_fit = np.atleast_2d(np.array(mean_affine_fit))
    
    # make the plot
    uap.plot_aligned_course_from_array(ix_master, 
                                       aligned_time_since_flash_arr, 
                                       aligned_course_arr, 
                                       aligned_roi_arr,
                                       aligned_flash_arr=None,
                                       aligned_linear_fits=None, # if None, skips plot
                                       aligned_affine_fits=mean_affine_fit, # if None, skips plot
                                       xlim_start=-300,
                                       xlim_end=300,
                                       ax=ax,
                                       clean_spines=True,
                                       course_marker_size=COURSE_MARKER_SIZE,
                                       course_alpha_multiplier=COURSE_ALPHA_MULTIPLIER,
                                       )

clean_labels(ax, show_labels)

item = layout.svgitems['text_n_' + windtype + '_' + flash_or_sham + '_unifying']
N_trajecs = len(unifying_algo_data.obj_id_unique_event.unique())
item.text = 'n=' + str(N_trajecs)

../../../Data/Unifying_Algo_Results/Supplemental/varwind_N700_flash_12.0_translationTrue.parquet


../../../Data/Unifying_Algo_Results/Supplemental/varwind_N700_flash_12.0_translationTrue.parquet
../../../Data/Experimental_Fly_Data/flies_varwind_C1WT_combined_optotrigger_fancontrol_trimmed.parquet


In [21]:
show_labels = False

In [22]:
windtype = '20'
unifying_algo_data, df = get_filenames_for_metadata_windtype(metadata, windtype)
if unifying_algo_data is not None:
    ax = layout.axes[(fifi_figure_label, windtype)]
    
    # build aligned arrays
    results = uaa.get_aligned_course_array(unifying_algo_data, df, return_linear_and_affine_fits=True)
    ix_master, aligned_time_since_flash_arr, aligned_course_arr, aligned_roi_arr, aligned_linear_fits, aligned_affine_fits = results
    
    # build flash arrays from aligned time
    flash_length = 500 # ms
    aligned_flash_arr = np.zeros_like(aligned_time_since_flash_arr)
    ix = np.where( (aligned_time_since_flash_arr>0) * (aligned_time_since_flash_arr<flash_length/1000.) )
    aligned_flash_arr[ix] = 1

    
    # Mean affine fit
    mean_affine_fit = [mean_angle(aligned_affine_fits[:,i]) for i in range(aligned_affine_fits.shape[1])]
    mean_affine_fit = np.array(mean_affine_fit)
    mean_affine_fit[0:200] = np.nan
    mean_affine_fit[400:] = np.nan
    mean_affine_fit = np.atleast_2d(np.array(mean_affine_fit))
    
    # make the plot
    uap.plot_aligned_course_from_array(ix_master, 
                                       aligned_time_since_flash_arr, 
                                       aligned_course_arr, 
                                       aligned_roi_arr,
                                       aligned_flash_arr=None,
                                       aligned_linear_fits=None, # if None, skips plot
                                       aligned_affine_fits=mean_affine_fit, # if None, skips plot
                                       xlim_start=-300,
                                       xlim_end=300,
                                       ax=ax,
                                       clean_spines=True,
                                       course_marker_size=COURSE_MARKER_SIZE,
                                       course_alpha_multiplier=COURSE_ALPHA_MULTIPLIER,
                                       )

clean_labels(ax, show_labels)

item = layout.svgitems['text_n_' + windtype + '_' + flash_or_sham + '_unifying']
N_trajecs = len(unifying_algo_data.obj_id_unique_event.unique())
item.text = 'n=' + str(N_trajecs)

../../../Data/Unifying_Algo_Results/Supplemental/varwind_N700_flash_20.0_translationTrue.parquet


../../../Data/Unifying_Algo_Results/Supplemental/varwind_N700_flash_20.0_translationTrue.parquet
../../../Data/Experimental_Fly_Data/flies_varwind_C1WT_combined_optotrigger_fancontrol_trimmed.parquet


In [23]:
windtype = '23'
unifying_algo_data, df = get_filenames_for_metadata_windtype(metadata, windtype)
if unifying_algo_data is not None:
    ax = layout.axes[(fifi_figure_label, windtype)]
    
    # build aligned arrays
    results = uaa.get_aligned_course_array(unifying_algo_data, df, return_linear_and_affine_fits=True)
    ix_master, aligned_time_since_flash_arr, aligned_course_arr, aligned_roi_arr, aligned_linear_fits, aligned_affine_fits = results
    
    # build flash arrays from aligned time
    flash_length = 500 # ms
    aligned_flash_arr = np.zeros_like(aligned_time_since_flash_arr)
    ix = np.where( (aligned_time_since_flash_arr>0) * (aligned_time_since_flash_arr<flash_length/1000.) )
    aligned_flash_arr[ix] = 1

    
    # Mean affine fit
    mean_affine_fit = [mean_angle(aligned_affine_fits[:,i]) for i in range(aligned_affine_fits.shape[1])]
    mean_affine_fit = np.array(mean_affine_fit)
    mean_affine_fit[0:200] = np.nan
    mean_affine_fit[400:] = np.nan
    mean_affine_fit = np.atleast_2d(np.array(mean_affine_fit))
    
    # make the plot
    uap.plot_aligned_course_from_array(ix_master, 
                                       aligned_time_since_flash_arr, 
                                       aligned_course_arr, 
                                       aligned_roi_arr,
                                       aligned_flash_arr=None,
                                       aligned_linear_fits=None, # if None, skips plot
                                       aligned_affine_fits=mean_affine_fit, # if None, skips plot
                                       xlim_start=-300,
                                       xlim_end=300,
                                       ax=ax,
                                       clean_spines=True,
                                       course_marker_size=COURSE_MARKER_SIZE,
                                       course_alpha_multiplier=COURSE_ALPHA_MULTIPLIER,
                                       )

clean_labels(ax, show_labels)

item = layout.svgitems['text_n_' + windtype + '_' + flash_or_sham + '_unifying']
N_trajecs = len(unifying_algo_data.obj_id_unique_event.unique())
item.text = 'n=' + str(N_trajecs)

../../../Data/Unifying_Algo_Results/Supplemental/varwind_N700_flash_23.0_translationTrue.parquet


../../../Data/Unifying_Algo_Results/Supplemental/varwind_N700_flash_23.0_translationTrue.parquet
../../../Data/Experimental_Fly_Data/flies_varwind_C1WT_combined_optotrigger_fancontrol_trimmed.parquet


In [24]:
windtype = '28'
unifying_algo_data, df = get_filenames_for_metadata_windtype(metadata, windtype)
if unifying_algo_data is not None:
    ax = layout.axes[(fifi_figure_label, windtype)]
    
    # build aligned arrays
    results = uaa.get_aligned_course_array(unifying_algo_data, df, return_linear_and_affine_fits=True)
    ix_master, aligned_time_since_flash_arr, aligned_course_arr, aligned_roi_arr, aligned_linear_fits, aligned_affine_fits = results
    
    # build flash arrays from aligned time
    flash_length = 500 # ms
    aligned_flash_arr = np.zeros_like(aligned_time_since_flash_arr)
    ix = np.where( (aligned_time_since_flash_arr>0) * (aligned_time_since_flash_arr<flash_length/1000.) )
    aligned_flash_arr[ix] = 1
    
    # Mean affine fit
    mean_affine_fit = [mean_angle(aligned_affine_fits[:,i]) for i in range(aligned_affine_fits.shape[1])]
    mean_affine_fit = np.array(mean_affine_fit)
    mean_affine_fit[0:200] = np.nan
    mean_affine_fit[400:] = np.nan
    mean_affine_fit = np.atleast_2d(np.array(mean_affine_fit))
    
    # make the plot
    uap.plot_aligned_course_from_array(ix_master, 
                                       aligned_time_since_flash_arr, 
                                       aligned_course_arr, 
                                       aligned_roi_arr,
                                       aligned_flash_arr=None,
                                       aligned_linear_fits=None, # if None, skips plot
                                       aligned_affine_fits=mean_affine_fit, # if None, skips plot
                                       xlim_start=-300,
                                       xlim_end=300,
                                       ax=ax,
                                       clean_spines=True,
                                       course_marker_size=COURSE_MARKER_SIZE,
                                       course_alpha_multiplier=COURSE_ALPHA_MULTIPLIER,
                                       )

clean_labels(ax, show_labels)

item = layout.svgitems['text_n_' + windtype + '_' + flash_or_sham + '_unifying']
N_trajecs = len(unifying_algo_data.obj_id_unique_event.unique())
item.text = 'n=' + str(N_trajecs)

../../../Data/Unifying_Algo_Results/Supplemental/varwind_N700_flash_28.0_translationTrue.parquet


../../../Data/Unifying_Algo_Results/Supplemental/varwind_N700_flash_28.0_translationTrue.parquet
../../../Data/Experimental_Fly_Data/flies_varwind_C1WT_combined_optotrigger_fancontrol_trimmed.parquet


In [25]:
windtype = '40'
unifying_algo_data, df = get_filenames_for_metadata_windtype(metadata, windtype)
if unifying_algo_data is not None:
    ax = layout.axes[(fifi_figure_label, windtype)]
    
    # build aligned arrays
    results = uaa.get_aligned_course_array(unifying_algo_data, df, return_linear_and_affine_fits=True)
    ix_master, aligned_time_since_flash_arr, aligned_course_arr, aligned_roi_arr, aligned_linear_fits, aligned_affine_fits = results
    
    # build flash arrays from aligned time
    flash_length = 500 # ms
    aligned_flash_arr = np.zeros_like(aligned_time_since_flash_arr)
    ix = np.where( (aligned_time_since_flash_arr>0) * (aligned_time_since_flash_arr<flash_length/1000.) )
    aligned_flash_arr[ix] = 1

    
    # Mean affine fit
    mean_affine_fit = [mean_angle(aligned_affine_fits[:,i]) for i in range(aligned_affine_fits.shape[1])]
    mean_affine_fit = np.array(mean_affine_fit)
    mean_affine_fit[0:200] = np.nan
    mean_affine_fit[400:] = np.nan
    mean_affine_fit = np.atleast_2d(np.array(mean_affine_fit))
    
    # make the plot
    uap.plot_aligned_course_from_array(ix_master, 
                                       aligned_time_since_flash_arr, 
                                       aligned_course_arr, 
                                       aligned_roi_arr,
                                       aligned_flash_arr=None,
                                       aligned_linear_fits=None, # if None, skips plot
                                       aligned_affine_fits=mean_affine_fit, # if None, skips plot
                                       xlim_start=-300,
                                       xlim_end=300,
                                       ax=ax,
                                       clean_spines=True,
                                       course_marker_size=COURSE_MARKER_SIZE,
                                       course_alpha_multiplier=COURSE_ALPHA_MULTIPLIER,
                                       )

clean_labels(ax, show_labels)

item = layout.svgitems['text_n_' + windtype + '_' + flash_or_sham + '_unifying']
N_trajecs = len(unifying_algo_data.obj_id_unique_event.unique())
item.text = 'n=' + str(N_trajecs)

../../../Data/Unifying_Algo_Results/Supplemental/varwind_N700_flash_40.0_translationTrue.parquet


../../../Data/Unifying_Algo_Results/Supplemental/varwind_N700_flash_40.0_translationTrue.parquet
../../../Data/Experimental_Fly_Data/flies_varwind_C1WT_combined_optotrigger_fancontrol_trimmed.parquet


In [26]:
windtype = '90'
unifying_algo_data, df = get_filenames_for_metadata_windtype(metadata, windtype)
if unifying_algo_data is not None:
    ax = layout.axes[(fifi_figure_label, windtype)]
    
    # build aligned arrays
    results = uaa.get_aligned_course_array(unifying_algo_data, df, return_linear_and_affine_fits=True)
    ix_master, aligned_time_since_flash_arr, aligned_course_arr, aligned_roi_arr, aligned_linear_fits, aligned_affine_fits = results
    
    # build flash arrays from aligned time
    flash_length = 500 # ms
    aligned_flash_arr = np.zeros_like(aligned_time_since_flash_arr)
    ix = np.where( (aligned_time_since_flash_arr>0) * (aligned_time_since_flash_arr<flash_length/1000.) )
    aligned_flash_arr[ix] = 1

    
    # Mean affine fit
    mean_affine_fit = [mean_angle(aligned_affine_fits[:,i]) for i in range(aligned_affine_fits.shape[1])]
    mean_affine_fit = np.array(mean_affine_fit)
    mean_affine_fit[0:200] = np.nan
    mean_affine_fit[400:] = np.nan
    mean_affine_fit = np.atleast_2d(np.array(mean_affine_fit))
    
    
    # make the plot
    uap.plot_aligned_course_from_array(ix_master, 
                                       aligned_time_since_flash_arr, 
                                       aligned_course_arr, 
                                       aligned_roi_arr,
                                       aligned_flash_arr=None,
                                       aligned_linear_fits=None, # if None, skips plot
                                       aligned_affine_fits=mean_affine_fit, # if None, skips plot
                                       xlim_start=-300,
                                       xlim_end=300,
                                       ax=ax,
                                       clean_spines=True,
                                       course_marker_size=COURSE_MARKER_SIZE,
                                       course_alpha_multiplier=COURSE_ALPHA_MULTIPLIER,
                                       )

clean_labels(ax, show_labels)

item = layout.svgitems['text_n_' + windtype + '_' + flash_or_sham + '_unifying']
N_trajecs = len(unifying_algo_data.obj_id_unique_event.unique())
item.text = 'n=' + str(N_trajecs)

../../../Data/Unifying_Algo_Results/Supplemental/varwind_N700_flash_90.0_translationTrue.parquet


../../../Data/Unifying_Algo_Results/Supplemental/varwind_N700_flash_90.0_translationTrue.parquet
../../../Data/Experimental_Fly_Data/flies_varwind_C1WT_combined_optotrigger_fancontrol_trimmed.parquet


In [27]:
layout.apply_svg_attrs()

layout.append_figure_to_layer(layout.figures[fifi_figure_label], fifi_figure_label, cleartarget=True)
layout.write_svg(FIGURE_NAME)

In [28]:
#data = diagnose_axis_elements(ax)